In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [ ]:
# fetch dataset
adult = fetch_ucirepo(id=2)
adult = adult.data.features.join(adult.data.targets, how="inner")

In [ ]:
adult.head(3)

## Basic Preprocessing Steps

### 1. Drop missing values

In [ ]:
# Drop missing values
adult.dropna(inplace=True)

### 2. Copy DataFrame for posterity

In [ ]:
df = adult.copy()

In [ ]:
adult["income"].value_counts()

### 3. Encode categorical variables

In [ ]:
def outcome_merge(val):
    if val == "<=50K" or val == "<=50K.":
        return 0
    else:
        return 1

In [ ]:
df["income"] = df["income"].apply(outcome_merge)

In [ ]:
#  sex, count and percentages above_50k

income_by_sex = df.groupby("sex")["income"].agg(
    ["count", lambda x: (x.sum() / x.count()) * 100]
)
income_by_sex.columns = ["count", "percentage_above_50k"]
income_by_sex

In [ ]:
#  race, count and percentages above_50k

income_by_race = df.groupby("race")["income"].agg(
    ["count", lambda x: (x.sum() / x.count()) * 100]
)
income_by_race.columns = ["count", "percentage_above_50k"]
income_by_race

In [ ]:
df["race"] = df["race"].replace("Amer-Indian-Eskimo", "Native American or Inuit")

### 4. Split the data

In [ ]:
# Split data
X = df.drop("income", axis=1)
y = df["income"]

In [ ]:
for col in X.columns:
    if isinstance(X[col], object):
        X[col] = X[col].astype("category")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [ ]:
y_train.value_counts()

## Train XGBoost Model

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    enable_categorical=True,
)
model.fit(X_train, y_train)

## Evaluate XGBoost Model

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)
print(classification_report(y_test, y_pred))

# Bias and Fairness Analysis with EquiBoots

**Equiboots supports a point estimate fairness analysis on a model's operating point (e.g., optimal threshold) as well as on multiple bootstraps with replacement.**


To initialize an analysis with equiboots:

1. Define a fairness Dataframe with the variables of interest.
2. Initialize an equiboots object using:
    - Ground truth (y_true)
    - Model probabilities (y_prob)
    - Model predictions (y_pred)
3. Identify the columns/variables that we will be assessing (e.g., race, sex)

In [ ]:
import equiboots as eqb

In [ ]:
fairness_df = X_test[["race", "sex"]].reset_index(drop=True)

In [ ]:
# get predictions and true values
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
y_test = y_test.to_numpy()

X_test[["race", "sex"]] = X_test[["race", "sex"]].astype(str)

## Point Estimates

In [ ]:
sensitive_features = ["race", "sex"]

fairness_df = X_test[sensitive_features].reset_index(drop=True)

eq = eqb.EquiBoots(
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    fairness_df=fairness_df,
    fairness_vars=sensitive_features,
)

eq.grouper(groupings_vars=sensitive_features)

In [ ]:
sliced_race_data = eq.slicer("race")
sliced_sex_data = eq.slicer("sex")

race_metrics = eq.get_metrics(sliced_race_data)
sex_metrics = eq.get_metrics(sliced_sex_data)

## Initial Set-up

### Step 1: Import and Initialize EquiBoots

In [ ]:
import equiboots as eqb

# Create fairness DataFrame
fairness_df = X_test[['race', 'sex']].reset_index()

eq = eqb.EquiBoots(
    y_true=y_test,
    y_prob=y_prob,
    y_pred=y_pred,
    fairness_df=fairness_df,
    fairness_vars=["race", "sex"],
)

In [ ]:
import equiboots as eqb

eq.grouper(groupings_vars=["race", "sex"])  
sliced_race_data = eq.slicer("race")
race_metrics = eq.get_metrics(sliced_race_data)

sliced_sex_data = eq.slicer("sex")
sex_metrics = eq.get_metrics(sliced_sex_data)

In [ ]:
import equiboots as eqb

race_metrics_df = eqb.metrics_dataframe(metrics_data=[race_metrics])
race_metrics_df = race_metrics_df[
    [
        "attribute_value",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Specificity",
        "TP Rate",
        "Prevalence",
        "Average Precision Score",
        "Calibration AUC",
    ]
]
## round to 3 decimal places for readability
round(race_metrics_df, 3)

In [ ]:
test_config = {
    "test_type": "chi_square",
    "alpha": 0.05,
    "adjust_method": "bonferroni",
    "classification_task": "binary_classification",
}
stat_test_results_race = eq.analyze_statistical_significance(
    race_metrics, "race", test_config
)

stat_test_results_sex = eq.analyze_statistical_significance(
    sex_metrics, "sex", test_config
)

overall_stat_results = {
    "sex": stat_test_results_sex,
    "race": stat_test_results_race,
}

In [ ]:
eq = eqb.EquiBoots(
    y_true=...,
    y_pred=...,
    y_prob=...,
    fairness_df=...,
    fairness_vars=["race", "sex"],
    reference_groups=["white", "female"]
)

In [ ]:
import equiboots
print(equiboots.__file__)

In [ ]:
eqb.eq_group_metrics_point_plot(
    group_metrics=[race_metrics, sex_metrics],
    metric_cols=[
        "Accuracy",
        "Precision",
        "Recall",
    ],
    category_names=["race", "sex"],
    figsize=(6, 8),
    include_legend=True,
    filename="Point_Disparity_Metrics.svg",
    raw_metrics=True,
    show_grid=True,
    y_lim=(0, 1.1),
    statistical_tests=overall_stat_results,
    show_pass_fail=False,
    show_reference=False,
    y_lims={(0, 0): (0.70, 1.0), (0, 1): (0.70, 1.0)},
    save_path="./images/",
)

In [ ]:
from equiboots.tables import metrics_table

stat_metrics_table_point = metrics_table(
    race_metrics,
    statistical_tests=stat_test_results_race,
    reference_group="White",
)

## Table with metrics per group and statistical significance shown on
## columns for omnibus and/or pairwise

round(stat_metrics_table_point,3)

In [ ]:
eqb.eq_plot_group_curves(
    sliced_race_data,
    filename="ROC_AUC_race_group.svg",
    curve_type="roc",
    title="ROC AUC by Race Group",
    figsize=(7, 7),
    decimal_places=2,
    subplots=False,
    exclude_groups=["Other"],
    save_path="./images/",
)

In [ ]:
eqb.eq_plot_group_curves(
    sliced_race_data,
    filename="PR_race_group.svg",
    curve_type="pr",
    subplots=False,
    figsize=(7, 7),
    title="Precision-Recall by Race Group",
    exclude_groups=["Other"],
    save_path="./images/",
)

In [ ]:
eqb.eq_plot_group_curves(
    sliced_race_data,
    filename="calibration_race_group.svg",
    curve_type="calibration",
    title="Calibration by Race Group",
    figsize=(7, 7),
    decimal_places=2,
    subplots=False,
    exclude_groups=["Other"],
    save_path="./images/",
)b

In [ ]:
eqb.eq_plot_group_curves(
    sliced_race_data,
    curve_type="calibration",
    filename="calibration_race_group_subplots.svg",
    title="Calibration by Race Group",
    figsize=(7, 7),
    decimal_places=2,
    subplots=True,
    shade_area=True,
    n_cols=1,
    exclude_groups=["Other"],
    save_path="./images/",
)

In [ ]:
eqb.eq_plot_group_curves(
    sliced_race_data,
    curve_type="calibration",
    title="Calibration by Race Group (LOWESS Smoothing)",
    filename="calibration_race_group_lowess.svg",
    figsize=(7, 7),
    decimal_places=2,
    subplots=True,
    lowess=0.6,
    lowess_kwargs={"linestyle": "--", "linewidth": 2, "alpha": 0.6},
    n_cols=1,
    exclude_groups=["Other"],
    save_path="./images/",
)

In [ ]:
eqb.eq_plot_group_curves(
    sliced_race_data,
    curve_type="calibration",
    title="Calibration by Race Group",
    filename="calibration_race_group_hist.svg",
    n_bins=10,
    n_cols=1,
    show_grid=False,
    exclude_groups=["Other"],
    plot_hist=True,
    save_path="./images/",

)

In [ ]:
eqb.eq_plot_metrics_forest(
    group_metrics=race_metrics,
    metric_name="Prevalence",
    filename="forest_plot_race_group.svg",
    title="Forest Plot: Race Group Point Estimates",
    reference_group="White",
    figsize=(8, 6),
    sort_groups=True,
    ascending=False,
    statistical_tests=stat_test_results_race,
    save_path="./images/",
)

In [ ]:
from equiboots import plot_effect_sizes

# Assume stat_results is a dict of group -> test results with .effect_size
plot_effect_sizes(
    xlabel="Attribute",
    stat_test_results=stat_test_results_race,
    ylabel="Effect size",
    title="Effect Sizes by Group",
    figsize=(8, 6),
    rotation=30,
    save_path="./images",
    filename="effect_sizes_demo.svg",
)

In [ ]:
fairness_df["age_group"] = age_group.reset_index(drop=True)

In [ ]:
eq = eqb.EquiBoots(
    y_true=y_test,
    y_prob=y_prob,
    y_pred=y_pred,
    fairness_df=fairness_df,
    fairness_vars=["race", "sex", "age_group"],
)
eq.grouper(groupings_vars=["race", "sex", "age_group"])

## Regression Residuals - Student Performance

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import equiboots as eqb

# Load dataset (UCI ID 320 = Student Performance)
student = fetch_ucirepo(id=320)
df = student.data.features.join(student.data.targets, how="inner")

# Target: final grade G3. Drop G1, G2 to avoid trivial leakage.
y = df["G3"]
X = df.drop(columns=["G1", "G2", "G3"])

# Encode string columns
for col in X.select_dtypes(include="object").columns:
    X[col] = LabelEncoder().fit_transform(X[col])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)

# Fit linear regression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_test = y_test.to_numpy()

# Bin age into groups for fairness slicing
age_group = pd.cut(
    X_test["age"],
    bins=[14, 16, 18, 20],
    labels=["15-16", "17-18", "19"],
    right=False,
    include_lowest=True,
).astype("str")

# Build fairness DataFrame
fairness_df = X_test[["sex", "address", "Pstatus"]].reset_index(drop=True)
fairness_df["age_group"] = age_group.reset_index(drop=True)

# Initialize EquiBoots in regression mode
eq = eqb.EquiBoots(
    y_true=y_test,
    y_pred=y_pred,
    fairness_df=fairness_df,
    fairness_vars=["sex", "address", "Pstatus", "age_group"],
    task="regression",
)
eq.grouper(groupings_vars=["sex", "address", "Pstatus", "age_group"])

# Slice residuals by age group
sliced_age_data = eq.slicer("age_group")

In [ ]:
eqb.eq_plot_residuals_by_group(
        data=sliced_age_data,
        filename="residuals_by_age_group_subplots.svg",
        title="Residuals by Age Group (Subplots)",
        color_by_group=True,
        show_centroids=True,
        show_grid=False,
        subplots=True,
        n_cols=1,
        save_path="./images/",
        figsize=(7, 6)
    )